In [1]:
# Cell 1: Install core dependencies
%pip install python-dotenv langchain langchain-openai langchain-community trafilatura spacy -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2: Load environment variables and verify keys are present
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env"
assert SERPER_API_KEY, "SERPER_API_KEY not found in .env"

print(f"OpenAI model : {OPENAI_MODEL}")
print(f"OpenAI key   : ...{OPENAI_API_KEY[-6:]}")
print(f"Serper key   : ...{SERPER_API_KEY[-6:]}")

OpenAI model : gpt-4o-mini
OpenAI key   : ...i-lLAA
Serper key   : ...d053d8


In [3]:
# Cell 3: Define the test category, its aliases, and trusted analyst domains
# Per doc1 §3 — gather under ALL aliases, not just the primary label
# Per doc1 §4 — use site-restricted searches, not open web

TEST_CATEGORY = "Account-Based Marketing"

CATEGORY_ALIASES = [
    "Account-Based Marketing",
    "ABM",
    "Account-Based Marketing Platforms",
    "ABM platforms",
    "Account-Based Everything",
    "ABX",
    "Account-Based Experience",
]

# Per doc2 — top analyst domains (sub-path level where it matters)
TRUSTED_SITES = [
    "blogs.gartner.com",
    "gartner.com/en/articles",
    "gartner.com/en/marketing/glossary",
    "gartner.com/en/information-technology/glossary",
    "forrester.com/blogs",
    "constellationr.com",
    "infotech.com",
    "kuppingercole.com",
    "hfsresearch.com",
    "everestgrp.com",
    "nucleusresearch.com",
    "isg-one.com",
    "gigaom.com",
    "aragonresearch.com",
]

print(f"Category     : {TEST_CATEGORY}")
print(f"Aliases      : {len(CATEGORY_ALIASES)}")
print(f"Trusted sites: {len(TRUSTED_SITES)}")

Category     : Account-Based Marketing
Aliases      : 7
Trusted sites: 14


In [7]:
# Cell 4: Serper search — batched (group sites with OR to reduce API calls)
import requests, time

def serper_search(query: str, api_key: str, num: int = 10) -> list[dict]:
    """Call Serper.dev Google Search API and return organic results."""
    resp = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
        json={"q": query, "num": num},
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json().get("organic", [])

def batch_site_queries(sites: list[str], batch_size: int = 5) -> list[str]:
    """Group sites into OR-joined site: clauses for fewer queries."""
    batches = []
    for i in range(0, len(sites), batch_size):
        chunk = sites[i : i + batch_size]
        clause = " OR ".join(f"site:{s}" for s in chunk)
        batches.append(f"({clause})")
    return batches

# Deduplicate aliases — keep only the most distinct search terms
SEARCH_ALIASES = [
    "Account-Based Marketing",       # primary
    "ABM platforms",                  # short-form + software
    "Account-Based Experience ABX",   # newer framing
]

site_batches = batch_site_queries(TRUSTED_SITES, batch_size=5)
total_queries = len(SEARCH_ALIASES) * len(site_batches)
print(f"Search plan: {len(SEARCH_ALIASES)} aliases × {len(site_batches)} site-batches = {total_queries} queries (was 98)")

seen_urls = set()
all_results = []

for alias in SEARCH_ALIASES:
    for site_clause in site_batches:
        query = f'{site_clause} "{alias}"'
        try:
            hits = serper_search(query, SERPER_API_KEY, num=10)
            for h in hits:
                url = h.get("link", "")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    all_results.append({
                        "url": url,
                        "title": h.get("title", ""),
                        "snippet": h.get("snippet", ""),
                        "query_alias": alias,
                    })
        except Exception as e:
            print(f"  ✗ {query[:70]}… → {e}")

print(f"\nTotal unique URLs discovered: {len(all_results)}")
for r in all_results[:15]:
    print(f"  • {r['title'][:80]}")
    print(f"    {r['url']}")

Search plan: 3 aliases × 3 site-batches = 9 queries (was 98)

Total unique URLs discovered: 34
  • The Account-Based Everything Framework - Gartner
    https://www.gartner.com/en/articles/the-account-based-everything-framework?0ecc245e_page=1
  • Product and Market Alignment FAQs | Resources for Vendor Portal
    https://gpivendorresources.gartner.com/en/articles/6812600-product-and-market-alignment-faqs
  • Build Your Account-Based Marketing Strategy - Info-Tech
    https://www.infotech.com/research/ss/build-your-account-based-marketing-strategy
  • Account-Based Marketing (ABM) Software - Info-Tech
    https://www.infotech.com/software-reviews/categories/account-based-marketing
  • Why B2B Sales Success Requires a Holistic Account-Based Strategy
    https://www.constellationr.com/research/why-b2b-sales-success-requires-holistic-account-based-strategy
  • Build Your Account-Based Marketing Strategy Workshop - Info-Tech
    https://www.infotech.com/workshops/build-your-account-based-ma

In [8]:
# Cell 5: Scrape discovered URLs with Trafilatura
# Extracts: clean text, title, author, date — strips boilerplate automatically
import trafilatura

def extract_article(url: str) -> dict | None:
    """Download and extract article content from a URL."""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return None
        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False,
            output_format="txt",
        )
        if not text:
            return None
        meta = trafilatura.metadata.extract_metadata(downloaded)
        return {
            "url": url,
            "title": meta.title if meta else "",
            "author": meta.author if meta else "",
            "date": meta.date if meta else "",
            "text": text,
            "hostname": meta.sitename if meta else "",
        }
    except Exception as e:
        return None

# Scrape all discovered URLs
scraped_sources = []
for i, r in enumerate(all_results):
    print(f"[{i+1}/{len(all_results)}] {r['url'][:80]}…", end=" ")
    article = extract_article(r["url"])
    if article and article["text"] and len(article["text"]) > 100:
        article["query_alias"] = r["query_alias"]
        scraped_sources.append(article)
        print(f"✓ ({len(article['text'])} chars)")
    else:
        print("✗ (skipped)")

print(f"\n{'='*50}")
print(f"Scraped {len(scraped_sources)} usable sources out of {len(all_results)} URLs")

[1/34] https://www.gartner.com/en/articles/the-account-based-everything-framework?0ecc2… ✗ (skipped)
[2/34] https://gpivendorresources.gartner.com/en/articles/6812600-product-and-market-al… ✓ (4011 chars)
[3/34] https://www.infotech.com/research/ss/build-your-account-based-marketing-strategy… ✓ (3057 chars)
[4/34] https://www.infotech.com/software-reviews/categories/account-based-marketing… ✓ (3618 chars)
[5/34] https://www.constellationr.com/research/why-b2b-sales-success-requires-holistic-… ✓ (1402 chars)
[6/34] https://www.infotech.com/workshops/build-your-account-based-marketing-strategy… ✓ (2714 chars)
[7/34] https://www.infotech.com/software-reviews/categories/account-based-marketing/com… ✓ (636 chars)
[8/34] https://www1.infotech.com/software-reviews/categories/account-based-marketing/co… ✓ (636 chars)
[9/34] https://www.infotech.com/software-reviews/categories/account-based-marketing/com… ✓ (398 chars)
[10/34] https://www.infotech.com/software-reviews/products/hubspot-abm?c_id=

In [9]:
# Cell 6: Preview scraped sources
for i, s in enumerate(scraped_sources):
    print(f"--- Source {i+1} ---")
    print(f"  Title  : {s['title']}")
    print(f"  Author : {s['author']}")
    print(f"  Date   : {s['date']}")
    print(f"  Host   : {s['hostname']}")
    print(f"  Alias  : {s['query_alias']}")
    print(f"  Length  : {len(s['text'])} chars")
    print(f"  Preview: {s['text'][:200]}…")
    print()

--- Source 1 ---
  Title  : Product and Market Alignment FAQs | Gartner Peer Insights | Resources for Vendor Portal
  Author : None
  Date   : 2025-04-14
  Host   : Gartner Peer Insights | Resources for Vendor Portal
  Alias  : Account-Based Marketing
  Length  : 4011 chars
  Preview: Which broad classification of markets and markets is available on Peer Insights?
We have created these new broad classifications to make it easier for you to find the right MQ/MG aligned markets and/o…

--- Source 2 ---
  Title  : Build Your Account-Based Marketing Strategy
  Author : Julie Geller
  Date   : 2022-11-22
  Host   : Build Your Account-Based Marketing Strategy | Info-Tech Research Group
  Alias  : Account-Based Marketing
  Length  : 3057 chars
  Preview: - Select the right ABM strategy for your organization and create complete alignment between the Sales and Marketing teams.
- Clear the clutter and jumpstart your ABM implementation with a solid founda…

--- Source 3 ---
  Title  : Account-Bas

In [10]:
# Cell 7: Pre-filter — remove review-platform & irrelevant pages per doc1/doc2 rules
# doc2: "SoftwareReviews is a separate Info-Tech property and is a review platform — drop"
# doc1: review sites, vendor comparison pages, SEO tutorial content → excluded

DROP_URL_PATTERNS = [
    "/software-reviews/",       # Info-Tech SoftwareReviews (review platform)
    "/compare/",                # head-to-head comparison pages
    "/products/",               # product review pages
    "gpivendorresources",       # Gartner vendor portal (vendor-facing, not analyst content)
]

def should_keep(source: dict) -> bool:
    url = source["url"].lower()
    for pattern in DROP_URL_PATTERNS:
        if pattern in url:
            return False
    if len(source["text"]) < 400:
        return False
    return True

filtered_sources = [s for s in scraped_sources if should_keep(s)]
dropped = len(scraped_sources) - len(filtered_sources)

print(f"Kept {len(filtered_sources)} sources, dropped {dropped}")
print()
for i, s in enumerate(filtered_sources):
    print(f"  {i+1}. [{s['hostname'] or '?'}] {s['title']}")
    print(f"     Author: {s['author'] or 'n/a'} | Date: {s['date'] or 'n/a'} | {len(s['text'])} chars")
    print(f"     {s['url']}")
    print()

Kept 16 sources, dropped 15

  1. [Build Your Account-Based Marketing Strategy | Info-Tech Research Group] Build Your Account-Based Marketing Strategy
     Author: Julie Geller | Date: 2022-11-22 | 3057 chars
     https://www.infotech.com/research/ss/build-your-account-based-marketing-strategy

  2. [Constellationr] Why B2B Sales Success Requires a Holistic Account-Based Strategy
     Author: Cindy Zhou | Date: 2017-09-09 | 1402 chars
     https://www.constellationr.com/research/why-b2b-sales-success-requires-holistic-account-based-strategy

  3. [Build Your Account-Based Marketing Strategy Workshop | Info-Tech Research Group] Build Your Account-Based Marketing Strategy
     Author: n/a | Date: 2018-01-01 | 2714 chars
     https://www.infotech.com/workshops/build-your-account-based-marketing-strategy

  4. [Constellationr] Constellation ShortList™ B2B Marketing Automation for Small and Midsize Business
     Author: Liz Miller | Date: 2025-08-08 | 4274 chars
     https://www.constellati

In [12]:
# Cell 8: LLM-based source quality scoring (per doc1 §5)
# Scores each source on: slot-fill, function-verbs, author credibility, currency, vendor diversity
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import json

class SourceScore(BaseModel):
    slot_definition: bool = Field(description="Contains a category definition")
    slot_capabilities: bool = Field(description="Lists core capabilities of the software")
    slot_boundaries: bool = Field(description="Distinguishes from adjacent/related categories")
    slot_buyer_use: bool = Field(description="Describes buyer persona or use case")
    slot_vendors: bool = Field(description="Names representative vendors/products")
    slots_filled: int = Field(description="Count of slots filled (0-5)")
    uses_function_verbs: bool = Field(description="Uses expert verbs like orchestrate, unify, score, route, match (vs SEO adjectives like better, smarter, faster)")
    vendor_count: int = Field(description="Number of distinct vendors/products mentioned")
    is_sme_content: bool = Field(description="Appears to be written by or for subject-matter experts, not SEO/marketing fluff")
    relevance_score: int = Field(description="1-10 overall relevance to defining the ABM software category")
    reasoning: str = Field(description="Brief explanation of the score")

SCORING_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are evaluating a web source for its usefulness in defining the software category "{category}".

Score it on these doc1 §5 criteria:
1. SLOT-FILL: Does it contain (a) a definition, (b) core capabilities, (c) boundaries vs adjacent categories, (d) buyer/use case, (e) representative vendors?
2. FUNCTION-VERBS: Does it use expert verbs (orchestrate, unify, score, route, match, segment, personalize, align) rather than SEO adjectives (better, smarter, faster, top, best)?
3. VENDOR DIVERSITY: How many distinct vendors are named? More = less biased.
4. SME CONTENT: Is this analyst/expert content or marketing fluff?
5. RELEVANCE: How useful is this source for writing a definitive category page (1-10)?

Return valid JSON matching the schema."""),
    ("human", """Source URL: {url}
Title: {title}
Author: {author}
Date: {date}
Host: {hostname}

Content (first 3000 chars):
{text}"""),
])

llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0, api_key=OPENAI_API_KEY)
structured_llm = llm.with_structured_output(SourceScore)
chain = SCORING_PROMPT | structured_llm

scored_sources = []
for i, s in enumerate(filtered_sources):
    print(f"[{i+1}/{len(filtered_sources)}] Scoring: {s['title'][:60]}…", end=" ")
    try:
        score = chain.invoke({
            "category": TEST_CATEGORY,
            "url": s["url"],
            "title": s["title"] or "Unknown",
            "author": s["author"] or "Unknown",
            "date": s["date"] or "Unknown",
            "hostname": s["hostname"] or "Unknown",
            "text": s["text"][:3000],
        })
        scored_sources.append({"source": s, "score": score})
        print(f"✓ relevance={score.relevance_score}/10, slots={score.slots_filled}/5")
    except Exception as e:
        print(f"✗ {e}")

# Sort by relevance score descending
scored_sources.sort(key=lambda x: x["score"].relevance_score, reverse=True)
print(f"\nScored {len(scored_sources)} sources. Top sources:")

[1/16] Scoring: Build Your Account-Based Marketing Strategy… ✓ relevance=8/10, slots=4/5
[2/16] Scoring: Why B2B Sales Success Requires a Holistic Account-Based Stra… ✓ relevance=8/10, slots=4/5
[3/16] Scoring: Build Your Account-Based Marketing Strategy… ✓ relevance=5/10, slots=4/5
[4/16] Scoring: Constellation ShortList™ B2B Marketing Automation for Small … ✓ relevance=7/10, slots=4/5
[5/16] Scoring: The Transformation of Account-Based Marketing for Revenue Op… ✓ relevance=9/10, slots=5/5
[6/16] Scoring: ABM Best Practices from The ON24 Experience - Aragon Researc… ✓ relevance=6/10, slots=3/5
[7/16] Scoring: Using External Data to Enhance Market Segmentation Research … ✓ relevance=3/10, slots=1/5
[8/16] Scoring: Marketing - ISG Research… ✓ relevance=4/10, slots=1/5
[9/16] Scoring: ISG Software Research Analyst Perspectives | Digital Marketi… ✓ relevance=2/10, slots=0/5
[10/16] Scoring: Marketing Performance Management - Ventana Research… ✓ relevance=4/10, slots=1/5
[11/16] Scoring: I

In [14]:
# Cell 9: Display scored sources ranked by relevance
for i, item in enumerate(scored_sources):
    s = item["source"]
    sc = item["score"]
    print(f"{'='*60}")
    print(f"#{i+1}  Relevance: {sc.relevance_score}/10 | Slots: {sc.slots_filled}/5 | Vendors mentioned: {sc.vendor_count}")
    print(f"  Title : {s['title']}")
    print(f"  Author: {s['author'] or 'n/a'} | Date: {s['date'] or 'n/a'} | Host: {s['hostname'] or 'n/a'}")
    print(f"  URL   : {s['url']}")
    print(f"  Slots → Def:{sc.slot_definition} Cap:{sc.slot_capabilities} Bound:{sc.slot_boundaries} Buyer:{sc.slot_buyer_use} Vendors:{sc.slot_vendors}")
    print(f"  Expert verbs: {sc.uses_function_verbs} | SME content: {sc.is_sme_content}")
    print(f"  Reasoning: {sc.reasoning}")
    print()

# Select top sources for synthesis (relevance >= 5 and at least 2 slots filled)
top_sources = [item for item in scored_sources if item["score"].relevance_score >= 5 and item["score"].slots_filled >= 2]
print(f"\n{'='*60}")
print(f"Sources qualifying for synthesis: {len(top_sources)} (relevance≥5, slots≥2)")

#1  Relevance: 9/10 | Slots: 5/5 | Vendors mentioned: 1
  Title : The Transformation of Account-Based Marketing for Revenue Optimization
  Author: Keith Dawson | Date: 2021-09-29 | Host: ISG Research
  URL   : https://research.isg-one.com/analyst-perspectives/the-transformation-of-account-based-marketing-for-revenue-optimization
  Slots → Def:True Cap:True Bound:True Buyer:True Vendors:True
  Expert verbs: True | SME content: True
  Reasoning: The source provides a comprehensive definition of Account-Based Marketing (ABM), outlines its core capabilities, distinguishes it from adjacent categories, describes the buyer persona, and mentions a representative vendor. It uses expert verbs effectively and appears to be written by a subject-matter expert, making it highly relevant for defining the ABM software category.

#2  Relevance: 8/10 | Slots: 4/5 | Vendors mentioned: 0
  Title : Build Your Account-Based Marketing Strategy
  Author: Julie Geller | Date: 2022-11-22 | Host: Build Your Acco

In [15]:
# Cell 10: Multi-source synthesis — produce the 5-slot category page (per doc1 §6)
# Uses LangChain refine-style approach: feed all top sources into one synthesis call

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class CategoryPage(BaseModel):
    category_name: str = Field(description="Primary category name")
    aliases: list[str] = Field(description="Known aliases for this category")
    definition: str = Field(description="2-4 sentence category definition synthesized from multiple sources")
    core_capabilities: list[str] = Field(description="Core software capabilities (use function-verbs)")
    boundaries: str = Field(description="What this category is NOT; how it differs from adjacent categories")
    buyer_use_case: str = Field(description="Who buys this software and why")
    representative_vendors: list[str] = Field(description="Named vendors from across sources")
    category_drift: str = Field(description="Where analyst firms disagree on scope, naming, or existence. Empty if no disagreement found.")
    source_count: int = Field(description="Number of sources used in synthesis")
    confidence: str = Field(description="high/medium/low — based on source coverage and consensus")

SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are writing a category definition page for Cuspera. The page must be written in Cuspera's editorial voice — no direct quotation from sources (copyright). Multi-source consensus carries the definition.

RULES (from doc1 §6):
- Fill five slots in order: definition, core capabilities, boundaries, buyer/use case, representative vendors.
- After the slots, address category drift if applicable: "the boundaries of this category are evolving, with [Firm A] framing it as X and [Firm B] framing it as Y."
- If analyst firms disagree on whether this is a distinct software market, acknowledge it directly.
- Use function-verbs (orchestrate, unify, score, route, segment, personalize) not benefit-adjectives.
- Capabilities should describe what the SOFTWARE does, not the methodology.
- Vendors must come from across sources, not a single source.

Category: {category}
Aliases: {aliases}"""),
    ("human", """Here are the top-scoring sources to synthesize from:

{sources_text}

Synthesize these into a single category page. Return structured JSON."""),
])

# Prepare source texts for the prompt
sources_block = ""
for i, item in enumerate(top_sources):
    s = item["source"]
    sc = item["score"]
    sources_block += f"\n--- SOURCE {i+1} (relevance {sc.relevance_score}/10, slots {sc.slots_filled}/5) ---\n"
    sources_block += f"Title: {s['title']}\n"
    sources_block += f"Author: {s['author'] or 'Unknown'} | Host: {s['hostname'] or 'Unknown'} | Date: {s['date'] or 'Unknown'}\n"
    sources_block += f"Content:\n{s['text'][:4000]}\n"

print(f"Synthesizing from {len(top_sources)} sources ({len(sources_block)} chars total)…\n")

synth_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2, api_key=OPENAI_API_KEY)
synth_chain = SYNTHESIS_PROMPT | synth_llm.with_structured_output(CategoryPage)

category_page = synth_chain.invoke({
    "category": TEST_CATEGORY,
    "aliases": ", ".join(CATEGORY_ALIASES),
    "sources_text": sources_block,
})

print("✓ Synthesis complete!")

Synthesizing from 6 sources (20426 chars total)…

✓ Synthesis complete!


In [19]:
# Cell 11: Display the final synthesized category page
print(f"{'='*70}")
print(f"  CATEGORY PAGE: {category_page.category_name}")
print(f"{'='*70}")
print(f"\nAliases: {', '.join(category_page.aliases)}")
print(f"Confidence: {category_page.confidence} | Sources used: {category_page.source_count}")

print(f"\n{'─'*70}")
print("1. DEFINITION")
print(f"{'─'*70}")
print(category_page.definition)

print(f"\n{'─'*70}")
print("2. CORE CAPABILITIES")
print(f"{'─'*70}")
for cap in category_page.core_capabilities:
    print(f"  • {cap}")

print(f"\n{'─'*70}")
print("3. BOUNDARIES")
print(f"{'─'*70}")
print(category_page.boundaries)

print(f"\n{'─'*70}")
print("4. BUYER / USE CASE")
print(f"{'─'*70}")
print(category_page.buyer_use_case)

print(f"\n{'─'*70}")
print("5. REPRESENTATIVE VENDORS")
print(f"{'─'*70}")
for v in category_page.representative_vendors:
    print(f"  • {v}")

if category_page.category_drift:
    print(f"\n{'─'*70}")
    print("6. CATEGORY DRIFT / ANALYST DISAGREEMENT")
    print(f"{'─'*70}")
    print(category_page.category_drift)

print(f"\n{'='*70}")
print("Sources used:")
for i, item in enumerate(top_sources):
    s = item["source"]
    print(f"  [{i+1}] {s['title']} — {s['author'] or 'n/a'} ({s['hostname'] or 'n/a'})")
    print(f"      {s['url']}")

  CATEGORY PAGE: Account-Based Marketing

Aliases: Account-Based Marketing, ABM, Account-Based Marketing Platforms, ABM platforms, Account-Based Everything, ABX, Account-Based Experience
Confidence: high | Sources used: 6

──────────────────────────────────────────────────────────────────────
1. DEFINITION
──────────────────────────────────────────────────────────────────────
Account-Based Marketing (ABM) is a strategic approach that focuses marketing efforts on a defined set of target accounts, aligning sales and marketing teams to deliver personalized campaigns. By leveraging data analytics and intent signals, ABM enables organizations to identify high-value accounts, engage decision-makers, and optimize marketing resources to drive revenue growth. This approach has evolved to encompass broader functions, integrating with marketing automation and CRM systems to enhance collaboration and effectiveness.

──────────────────────────────────────────────────────────────────────
2. CORE CAP